# SQL: Дополнение 2 (Продвинутые темы)

В этом ноутбуке мы расширим знания, полученные в ходе теста, и разберем инструменты, которые делают SQL по-настоящему мощным: CTE, сложные оконные функции и условную логику.

## 1. CTE (Common Table Expressions) — Обобщенные табличные выражения

**Для чего:** Позволяют создавать временные результирующие наборы, которые можно использовать внутри одного запроса. Это делает код чище и заменяет вложенные подзапросы.

**Синтаксис:**
```sql
WITH temp_table AS (
    SELECT ...
)
SELECT * FROM temp_table;
```

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.executescript('''
CREATE TABLE Employees (id INTEGER, name TEXT, salary INTEGER, dept_id INTEGER);
INSERT INTO Employees VALUES (1, 'Иван', 50000, 1), (2, 'Мария', 70000, 1), (3, 'Петр', 40000, 2), (4, 'Анна', 90000, 2);
''')

# Пример CTE: Находим сотрудников с зарплатой выше средней
query = '''
WITH AvgSalary AS (
    SELECT AVG(salary) as av_sal FROM Employees
)
SELECT * FROM Employees, AvgSalary
WHERE salary > av_sal
'''
pd.read_sql_query(query, conn)

,id,name,salary,dept_id,av_sal
0,2,Мария,70000,1,62500.0
1,4,Анна,90000,2,62500.0


## 2. Разница между ROW_NUMBER, RANK и DENSE_RANK

В тесте мы упоминали оконные функции. Чаще всего они используются для нумерации строк:
- `ROW_NUMBER()`: просто нумерует строки подряд (1, 2, 3, 4).
- `RANK()`: если значения одинаковые, ставит одинаковый ранг, но **пропускает** следующий (1, 2, 2, 4).
- `DENSE_RANK()`: ставит одинаковый ранг, но **не пропускает** следующий (1, 2, 2, 3).

In [3]:
# Добавим дубликат по зарплате для наглядности
cursor.execute("INSERT INTO Employees VALUES (5, 'Сергей', 70000, 1)")

query = '''
SELECT name, salary, 
       ROW_NUMBER() OVER(ORDER BY salary DESC) as row_num,
       RANK() OVER(ORDER BY salary DESC) as rnk,
       DENSE_RANK() OVER(ORDER BY salary DESC) as dense_rnk
FROM Employees
'''
pd.read_sql_query(query, conn)

,name,salary,row_num,rnk,dense_rnk
0,Анна,90000,1,1,1
1,Мария,70000,2,2,2
2,Сергей,70000,3,2,2
3,Иван,50000,4,4,3
4,Петр,40000,5,5,4


## 3. Оператор CASE (Условная логика)

Аналог `if-else` в SQL. Позволяет создавать новые колонки на основе условий.

In [4]:
query = '''
SELECT name, salary,
       CASE 
           WHEN salary >= 70000 THEN 'Высокая'
           WHEN salary >= 50000 THEN 'Средняя'
           ELSE 'Низкая'
       END as salary_level
FROM Employees
'''
pd.read_sql_query(query, conn)

,name,salary,salary_level
0,Иван,50000,Средняя
1,Мария,70000,Высокая
2,Петр,40000,Низкая
3,Анна,90000,Высокая
4,Сергей,70000,Высокая


## 4. EXCEPT и INTERSECT (Множества)

В дополнение к `UNION`:
- **INTERSECT**: возвращает только те строки, которые есть в **обоих** запросах.
- **EXCEPT**: возвращает строки из первого запроса, которых **нет** во втором.

## Практические задачи

### Задача 1 (CTE + Агрегация)
Используя CTE, найдите департамент (`dept_id`) с самой высокой суммарной зарплатой.

### Задача 2 (CASE)
Выведите имена сотрудников, добавив колонку, где написано 'Бонус', если зарплата < 50000, и 'Ок', если >= 50000.

### Задача 3 (Оконные функции)
Пронумеруйте сотрудников внутри каждого департамента по убыванию зарплаты (используйте `PARTITION BY`).

In [5]:
# Задача 1: Департамент с самой высокой суммарной зарплатой
query = '''
WITH DeptSalaries AS (
    SELECT dept_id, SUM(salary) as total_salary
    FROM Employees
    GROUP BY dept_id
)
SELECT dept_id, total_salary
FROM DeptSalaries
ORDER BY total_salary DESC
LIMIT 1
'''
pd.read_sql_query(query, conn)

,dept_id,total_salary
0,1,190000


In [6]:
# Задача 2: Имена сотрудников с пометкой о бонусе
query = '''
SELECT name, salary,
       CASE 
           WHEN salary < 50000 THEN 'Бонус'
           ELSE 'Ок'
       END as bonus_status
FROM Employees
'''
pd.read_sql_query(query, conn)

,name,salary,bonus_status
0,Иван,50000,Ок
1,Мария,70000,Ок
2,Петр,40000,Бонус
3,Анна,90000,Ок
4,Сергей,70000,Ок


In [7]:
# Задача 3: Нумерация сотрудников внутри департаментов
query = '''
SELECT name, dept_id, salary,
       ROW_NUMBER() OVER(PARTITION BY dept_id ORDER BY salary DESC) as rank_in_dept
FROM Employees
'''
pd.read_sql_query(query, conn)

,name,dept_id,salary,rank_in_dept
0,Мария,1,70000,1
1,Сергей,1,70000,2
2,Иван,1,50000,3
3,Анна,2,90000,1
4,Петр,2,40000,2
